# Preprocessing notebook after discovery

## 1. Import Libraries & Setup

In [79]:
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


ROOT = pathlib.Path.cwd().parent
DATA_PATH = ROOT / "data/ieee-fraud-detection"
ID_COL = "TransactionID"
TARGET = "isFraud"
CORE_COLUMNS = [
    "isFraud",
    "TransactionID",
    "TransactionDT",
    "TransactionAmt",
    "ProductCD",
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
]


pd.set_option("display.max_colwidth", None)

plt.style.use("ggplot")

## 2. Loading the data

In [89]:
raw_df_train = pd.read_csv(DATA_PATH / "train_transaction.csv")
df_train = raw_df_train[CORE_COLUMNS].drop_duplicates().copy()

print(f"raw: {raw_df_train.shape} | core: {df_train.shape}")

raw: (590540, 394) | core: (590540, 11)


## 3. Imports from src

In [81]:
import sys

sys.path.append("..")

from src.preprocessing import (
    CategoryEncoder,
    DataCleaner,
    Log1pTransformer,
    TimeDeltaFeatures,
)

## 4. Time-based split

In [90]:
df_train = df_train.sort_values("TransactionDT").reset_index(drop=True)

X = df_train.drop(columns=[TARGET, ID_COL])
y = df_train[TARGET]

cut = int(len(df_train) * 0.8)
X_train, X_test = X.iloc[:cut], X.iloc[cut:]
y_train, y_test = y.iloc[:cut], y.iloc[cut:]

print(f"train: {X_train.shape} | test: {X_test.shape}")

train: (472432, 9) | test: (118108, 9)


## 5. Pipeline

In [91]:
from sklearn.pipeline import Pipeline

pipe = Pipeline(
    [
        ("clean", DataCleaner(drop_duplicates=False, drop_null_rows=False)),
        ("time", TimeDeltaFeatures()),
        ("log", Log1pTransformer(columns=["TransactionAmt","card1",'card2','card3','card5'])),
        ("cat", CategoryEncoder(columns=["ProductCD", "card4", "card6"])),
    ]
)

X_train_t = pipe.fit_transform(X_train)
X_test_t = pipe.transform(X_test)

print(X_train_t.shape, X_test_t.shape)

(472432, 11) (118108, 11)


## 6. Save

In [92]:
import joblib

OUT = ROOT / "artifacts"
(OUT / "preprocessed").mkdir(parents=True, exist_ok=True)

joblib.dump(pipe, OUT / "pipeline.joblib")

X_train_t.to_parquet(OUT / "preprocessed/X_train.parquet")
X_test_t.to_parquet(OUT / "preprocessed/X_test.parquet")
y_train.to_frame().to_parquet(OUT / "preprocessed/y_train.parquet")
y_test.to_frame().to_parquet(OUT / "preprocessed/y_test.parquet")